In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("juhibhojani/hotel-reviews")

print("Downloaded to:", path)
print("Files:", os.listdir(path))

100%|██████████| 319k/319k [00:00<00:00, 474kB/s]

Extracting files...
Downloaded to: C:\Users\yasan\.cache\kagglehub\datasets\juhibhojani\hotel-reviews\versions\1
Files: ['hotel_reviews.csv']


In [ ]:
import shutil
import os

source = r"C:\Users\yasan\.cache\kagglehub\datasets\juhibhojani\hotel-reviews\versions\1\hotel_reviews.csv"

destination = r"C:\Users\yasan\Desktop\DL Assignment\customer-feedback-classification\data\raw\hotel_reviews.csv"

shutil.copy2(source, destination)

print("Dataset copied to:")
print(destination)

In [ ]:
import shutil
import os

source = os.path.join(path, "hotel_reviews.csv")
destination = r"C:\Users\yasan\Desktop\DL Assignment\customer-feedback-classification\data\raw\hotel_reviews.csv"

shutil.copy2(source, destination)

print("Dataset copied to:")
print(destination)

In [2]:
import shutil
import os

source = r"C:\Users\yasan\.cache\kagglehub\datasets\juhibhojani\hotel-reviews\versions\1\hotel_reviews.csv"

destination = r"C:\Users\yasan\Desktop\DL Assignment\customer-feedback-classification\data\raw\hotel_reviews.csv"

shutil.copy2(source, destination)

print("Dataset copied to:")
print(destination)

Dataset copied to:
C:\Users\yasan\Desktop\DL Assignment\customer-feedback-classification\data\raw\hotel_reviews.csv


In [3]:
import pandas as pd

file_path = r"C:\Users\yasan\Desktop\DL Assignment\customer-feedback-classification\data\raw\hotel_reviews.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (7001, 7)

Columns:
['Index', 'Name', 'Area', 'Review_Date', 'Rating_attribute', 'Rating(Out of 10)', 'Review_Text']

First 5 rows:


,Index,Name,Area,Review_Date,Rating_attribute,Rating(Out of 10),Review_Text
0,0,Hotel The Pearl,"Paharganj, New Delhi",Jul-23,Best budget friendly hotel,9.0,Hotel the pearl is perfect place to stay in De...
1,1,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Amazing place,9.0,Location of the hotel is perfect. The hotel is...
2,2,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Overall good stay. Economic.,9.0,"Location, Indian food."
3,3,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Lovely,9.0,The location and the hotel itself is great. Ne...
4,4,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Great hotel Great staff and great staying,9.0,Friendly and smiling staffs.. The reception st...


In [4]:
print("Rating distribution:")
print(df["Rating(Out of 10)"].value_counts().sort_index())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Rating distribution:
Rating(Out of 10)
1.0      866
2.0      119
2.5        2
3.0      124
4.0      133
5.0      320
6.0      493
7.0      959
7.9        1
8.0     1360
9.0     1188
10.0    1436
Name: count, dtype: int64

Missing values:
Index                0
Name                 0
Area                 0
Review_Date          0
Rating_attribute     0
Rating(Out of 10)    0
Review_Text          7
dtype: int64

Duplicate rows: 0


In [5]:
# Remove reviews with missing text
df_clean = df.dropna(subset=["Review_Text"]).copy()

# Create 5 rating categories
def rating_category(rating):
    if 1 <= rating <= 2:
        return "Terrible"
    elif 3 <= rating <= 4:
        return "Poor"
    elif 5 <= rating <= 6:
        return "Average"
    elif 7 <= rating <= 8:
        return "Good"
    elif 9 <= rating <= 10:
        return "Great"

df_clean["Category"] = df_clean["Rating(Out of 10)"].apply(rating_category)

# Numeric labels for the neural networks
label_mapping = {
    "Terrible": 0,
    "Poor": 1,
    "Average": 2,
    "Good": 3,
    "Great": 4
}

df_clean["label"] = df_clean["Category"].map(label_mapping)

print("Dataset size:", len(df_clean))

print("\nCategory distribution:")
print(df_clean["Category"].value_counts())

print("\nCategory percentages:")
print(
    df_clean["Category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Dataset size: 6994

Category distribution:
Category
Great       2624
Good        2318
Terrible     982
Average      811
Poor         257
Name: count, dtype: int64

Category percentages:
Category
Great       37.53
Good        33.15
Terrible    14.04
Average     11.60
Poor         3.68
Name: proportion, dtype: float64


In [6]:
# Remove reviews with missing text
df_clean = df.dropna(subset=["Review_Text"]).copy()

# Create 3 rating categories
def rating_category(rating):
    if rating < 5:
        return "Poor"
    elif rating < 8:
        return "Average"
    else:
        return "Good"

df_clean["Category"] = df_clean["Rating(Out of 10)"].apply(rating_category)

# Convert categories to numerical labels
label_mapping = {
    "Poor": 0,
    "Average": 1,
    "Good": 2
}

df_clean["label"] = df_clean["Category"].map(label_mapping)

# Show results
print("Dataset size:", len(df_clean))

print("\nCategory distribution:")
print(df_clean["Category"].value_counts())

print("\nCategory percentages:")
print(
    df_clean["Category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Dataset size: 6994

Category distribution:
Category
Good       3984
Average    1769
Poor       1241
Name: count, dtype: int64

Category percentages:
Category
Good       56.96
Average    25.29
Poor       17.74
Name: proportion, dtype: float64


In [7]:
from sklearn.model_selection import train_test_split

# Features and labels
X = df_clean["Review_Text"]
y = df_clean["label"]

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide the temporary 30% into
# 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display sizes
print("Training set:", len(X_train))
print("Validation set:", len(X_val))
print("Test set:", len(X_test))

Training set: 4895
Validation set: 1049
Test set: 1050


In [8]:
print("Training distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).mul(100).round(2))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training distribution:
label
2    56.96
1    25.29
0    17.75
Name: proportion, dtype: float64

Validation distribution:
label
2    57.01
1    25.26
0    17.73
Name: proportion, dtype: float64

Test distribution:
label
2    56.95
1    25.33
0    17.71
Name: proportion, dtype: float64


In [9]:
# Display a few training reviews before preprocessing

for i in range(3):
    print(f"Review {i+1}:")
    print(X_train.iloc[i])
    print("-" * 80)

Review 1:
Breakfast is very good and staff is so helpfull
--------------------------------------------------------------------------------
Review 2:
Booking ke baad bolte he,,hume nahi ptaa yahan aake book karna hogaa....
--------------------------------------------------------------------------------
Review 3:
I recently stayed at Fab Hotel Jasmine with my family while my father was undergoing treatment at the nearby Apollo Hospital. I must say that our experience at this hotel was nothing short of exceptional. From the moment we arrived, the staff at Fab Hotel Jasmine displayed a level of consideration and genuine care that was truly heartwarming. They went above and beyond to ensure that we felt comfortable, safe, and well taken care of during our stay. Given the stressful circumstances surrounding our visit, their attentiveness made a significant difference and put us at ease.
One aspect that particularly impressed us was the cleanliness of the hotel. The housekeeping staff provide

In [10]:
import re

def clean_text(text):
    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply the same preprocessing to all three sets
X_train_clean = X_train.apply(clean_text)
X_val_clean = X_val.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# Display examples
for i in range(3):
    print(f"Cleaned Review {i+1}:")
    print(X_train_clean.iloc[i])
    print("-" * 80)

Cleaned Review 1:
breakfast is very good and staff is so helpfull
--------------------------------------------------------------------------------
Cleaned Review 2:
booking ke baad bolte he,,hume nahi ptaa yahan aake book karna hogaa....
--------------------------------------------------------------------------------
Cleaned Review 3:
i recently stayed at fab hotel jasmine with my family while my father was undergoing treatment at the nearby apollo hospital. i must say that our experience at this hotel was nothing short of exceptional. from the moment we arrived, the staff at fab hotel jasmine displayed a level of consideration and genuine care that was truly heartwarming. they went above and beyond to ensure that we felt comfortable, safe, and well taken care of during our stay. given the stressful circumstances surrounding our visit, their attentiveness made a significant difference and put us at ease. one aspect that particularly impressed us was the cleanliness of the hotel. the ho

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizer settings
MAX_WORDS = 10000
MAX_LENGTH = 200

# Create tokenizer
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# IMPORTANT: fit only on training data
tokenizer.fit_on_texts(X_train_clean)

# Convert text to integer sequences
X_train_seq = tokenizer.texts_to_sequences(X_train_clean)
X_val_seq = tokenizer.texts_to_sequences(X_val_clean)
X_test_seq = tokenizer.texts_to_sequences(X_test_clean)

# Pad sequences to the same length
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)
print("Test shape:", X_test_pad.shape)

Training shape: (4895, 200)
Validation shape: (1049, 200)
Test shape: (1050, 200)


In [12]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Calculate class weights using training data only
classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(
    zip(classes, class_weights_array)
)

print("Class weights:")
for label, weight in class_weights.items():
    print(f"Label {label}: {weight:.4f}")

Class weights:
Label 0: 1.8776
Label 1: 1.3180
Label 2: 0.5852


In [13]:
import os
import pickle

# Create processed data directory
os.makedirs("../data/processed", exist_ok=True)

# Save padded datasets and labels
np.save("../data/processed/X_train_pad.npy", X_train_pad)
np.save("../data/processed/X_val_pad.npy", X_val_pad)
np.save("../data/processed/X_test_pad.npy", X_test_pad)

np.save("../data/processed/y_train.npy", y_train.to_numpy())
np.save("../data/processed/y_val.npy", y_val.to_numpy())
np.save("../data/processed/y_test.npy", y_test.to_numpy())

# Save tokenizer
with open("../data/processed/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save class weights
with open("../data/processed/class_weights.pkl", "wb") as f:
    pickle.dump(class_weights, f)

print("Processed data and preprocessing objects saved successfully.")

Processed data and preprocessing objects saved successfully.
